# Corporate Bond Maturity Wall Stress — Exploratory Analysis

**Strategy:** Corporate Bond Maturity Wall Stress via CUSIP-Level Refinancing Calendar  
**Notebook:** 01 — Full pipeline walkthrough: data acquisition → signal construction → backtest  

---

**Navigation**
1. Universe Overview
2. XBRL Data Acquisition Demo
3. XBRL Quality Audit
4. Maturity Schedule Construction
5. TRACE Bond Stack Enrichment
6. Signal Component Analysis
7. Macro Rate Overlay
8. Signal Validation: Historical Stress Events
9. Statistical Tests
10. Backtest Results
11. Limitations & Extensions

In [ ]:
import sys
sys.path.insert(0, '..')  # allow importing from repo root

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import config
from src.utils import setup_logging

logger = setup_logging()
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', font_scale=1.1)

print('✓ Imports complete | REPO_ROOT:', config.REPO_ROOT)

---
## Section 1: Universe Overview

The seed universe contains 50 companies selected for high leverage, bond market reliance, and historical or potential maturity-wall stress.  
Selection criteria:
- Issued publicly traded bonds (not bank-only borrowers)
- Minimum $100MM total debt (`config.MIN_DEBT_SIZE_MM`)
- Sufficient XBRL filing history (at least one 10-K after 2014)
- Mix of confirmed stress events (validation) and potential future events (forward signal)

In [ ]:
universe_df = pd.read_csv(config.COMPANY_UNIVERSE_CSV)
print(f'Universe: {len(universe_df)} companies | Included: {universe_df["include_flag"].sum()}')
display(universe_df.head(10))

In [ ]:
# Sector breakdown
sector_summary = universe_df.groupby('sector').agg(
    n_companies=('ticker', 'count'),
    avg_debt_bn=('avg_total_debt_bn', 'mean'),
    known_stress_events=('known_stress_event', 'sum'),
).sort_values('n_companies', ascending=False)

print('\n--- Sector Breakdown ---')
display(sector_summary)

# Debt category mix
print('\n--- Debt Category Mix ---')
display(universe_df['debt_category'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Sector distribution
sector_summary['n_companies'].plot(kind='bar', ax=axes[0], color='#3498DB')
axes[0].set_title('Companies by Sector', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Debt category
universe_df['debt_category'].value_counts().plot(kind='pie', ax=axes[1],
    autopct='%1.0f%%', colors=['#E74C3C', '#3498DB', '#27AE60'])
axes[1].set_title('Debt Category Mix', fontweight='bold')
axes[1].set_ylabel('')

# Avg debt distribution
axes[2].hist(universe_df['avg_total_debt_bn'].dropna(), bins=15, color='#27AE60', edgecolor='white')
axes[2].set_title('Avg Total Debt Distribution ($bn)', fontweight='bold')
axes[2].set_xlabel('Avg Total Debt ($bn)')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()
print(f'Why high-yield and leveraged issuers? Because the maturity wall thesis is a '
      f'"refinancing cliff" problem — only relevant when refinancing costs matter.\n'
      f'Investment-grade issuers have essentially unlimited market access at any rate level.\n'
      f'HY and leveraged issuers face genuine binary risk: roll the debt or default.')

---
## Section 2: XBRL Data Acquisition Demo

EDGAR's `/api/xbrl/companyfacts/CIK{cik}.json` endpoint returns every XBRL-tagged fact across all filings for a company.  
A single call returns the full history — no pagination.  
File size: typically 5–20MB for a large filer with 10+ years of XBRL history.

> **Note:** Running these cells will make live EDGAR API calls (rate-limited to 10 req/sec).  
> Results are cached to `data/raw/xbrl/` after the first call.

In [ ]:
from src.data_loader import fetch_company_xbrl_facts
from src.xbrl_parser import extract_debt_maturity_facts

tag_taxonomy = pd.read_csv(config.XBRL_TAG_TAXONOMY_CSV)
print(f'Tag taxonomy loaded: {len(tag_taxonomy)} entries')
print(f'Standard tags: {tag_taxonomy["is_standard"].sum()} | Extension tags: {tag_taxonomy["is_extension"].sum()}')
display(tag_taxonomy.head(8))

In [ ]:
# Demo: HCA Healthcare (CIK 860731) — large HY healthcare issuer
HCA_CIK = '0000860731'

hca_facts = fetch_company_xbrl_facts(HCA_CIK)
print(f"\nEntity: {hca_facts.get('entityName')}")
print(f"Namespaces: {list(hca_facts.get('facts', {}).keys())}")

# Count facts by namespace
for ns, tags in hca_facts.get('facts', {}).items():
    n_tags = len(tags)
    n_obs  = sum(len(v.get('units', {}).get('USD', [])) for v in tags.values())
    print(f"  {ns}: {n_tags} tags, {n_obs} USD observations")

In [ ]:
# Extract debt maturity facts for HCA
hca_raw_facts = extract_debt_maturity_facts(hca_facts, tag_taxonomy)
print(f'\nRaw maturity facts extracted: {len(hca_raw_facts)} rows')
print(f'Standard tags found: {hca_raw_facts["is_standard_tag"].sum()}')
print(f'Extension tags found: {(~hca_raw_facts["is_standard_tag"]).sum()}')
print(f'Unique tag names: {hca_raw_facts["tag_name"].nunique()}')
print(f'\nTag-to-bucket mapping:')
display(hca_raw_facts.groupby(['tag_name', 'maturity_bucket_raw']).size().reset_index(name='count').head(15))

In [ ]:
# Show UNKNOWN rate — data quality KPI
n_unknown = (hca_raw_facts['maturity_bucket_raw'] == 'UNKNOWN').sum()
pct_unknown = 100 * n_unknown / len(hca_raw_facts) if len(hca_raw_facts) > 0 else 0
print(f'\n[DATA QUALITY KPI] UNKNOWN bucket rate: {pct_unknown:.1f}% (target < 10%)')
if pct_unknown > 10:
    print('  ⚠️  UNKNOWN rate exceeds threshold — taxonomy needs extension')
else:
    print('  ✓ UNKNOWN rate within acceptable range')

---
## Section 3: XBRL Quality Audit

The coverage quality audit compares the sum of the XBRL maturity schedule against the LongTermDebt balance-sheet total.  
A coverage ratio of 0.3 means we are capturing only 30% of the debt maturity profile — the MCS computed from this data will systematically understate concentration risk.

In [ ]:
from src.xbrl_parser import build_annual_maturity_schedule, validate_xbrl_against_10k_text

hca_annual = build_annual_maturity_schedule(HCA_CIK, hca_raw_facts, pd.DataFrame())
print(f'Annual maturity schedule: {len(hca_annual)} 10-K filings')
print(f'Coverage flag distribution:')
display(hca_annual['coverage_flag'].value_counts())

display_cols = ['filing_date', 'Y1', 'Y2', 'Y3', 'Y4', 'Y5', 'Y5plus', 'total_reported', 'coverage_flag']
display_cols = [c for c in display_cols if c in hca_annual.columns]
display(hca_annual[display_cols].tail(8))

In [ ]:
hca_validation = validate_xbrl_against_10k_text(HCA_CIK, hca_annual, hca_facts)
print('\n=== XBRL Validation Report: HCA Healthcare ===')
for k, v in hca_validation.items():
    print(f'  {k}: {v}')

In [ ]:
# Universe quality report (if available from prior run)
import os
if config.XBRL_QUALITY_REPORT_CSV.exists():
    quality_df = pd.read_csv(config.XBRL_QUALITY_REPORT_CSV)
    print(f'Universe XBRL quality report: {len(quality_df)} companies')
    print('\nRecommendation distribution:')
    display(quality_df['recommendation'].value_counts())
    print('\nWorst coverage companies (lowest mean_coverage_ratio):')
    display(quality_df.nsmallest(5, 'mean_coverage_ratio')[['ticker', 'mean_coverage_ratio', 'recommendation']])

    from src.visualize import plot_xbrl_coverage_quality_report
    fig = plot_xbrl_coverage_quality_report(quality_df, universe_df)
    plt.show()
else:
    print('Quality report not yet generated. Run build_universe_maturity_schedules() first.')

---
## Section 4: Maturity Schedule Construction

The annual XBRL buckets (Y1–Y5+) are distributed evenly across 4 quarters each.  
This is a deliberate simplification: Y1 = all debt maturing in the next 12 months, divided by 4.  

**Information lost:** A company with $1bn maturing in a single month (one bond) is treated the same as one with $250mm maturing each quarter. The TRACE enrichment path corrects this when CUSIP-level data is available.

In [ ]:
from src.xbrl_parser import convert_annual_to_quarterly_schedule

hca_quarterly = convert_annual_to_quarterly_schedule(hca_annual)
print(f'Quarterly schedule: {len(hca_quarterly)} rows')
display(hca_quarterly.tail(12))

In [ ]:
from src.visualize import plot_maturity_wall_waterfall

hca_quarterly['ticker'] = 'HCA'
as_of = pd.Timestamp('2023-01-01')

fig = plot_maturity_wall_waterfall('HCA', hca_quarterly, as_of)
plt.show()
print('Figure 1: HCA debt maturity wall. One look tells you everything about the '
      'binary refinancing risk. Peak quarter highlighted in red.')

---
## Section 5: TRACE Bond Stack Enrichment

FINRA TRACE bond reference data provides: issuer, coupon rate, maturity date, outstanding face value.  
This is used to:  
1. Assign precise quarterly maturity dates (replacing the even-distribution approximation)
2. Provide bond-specific coupon rates for the RDM yield_at_issuance proxy

In [ ]:
from src.trace_parser import build_cusip_list_from_edgar, build_issuer_bond_stack, compute_weighted_avg_coupon

# Demo: discover CUSIPs for HCA
hca_cusips = build_cusip_list_from_edgar(HCA_CIK)
print(f'HCA CUSIPs discovered from EDGAR filings: {len(hca_cusips)}')
print(hca_cusips[:10])

In [ ]:
hca_bond_stack = build_issuer_bond_stack(HCA_CIK, 'HCA')
if not hca_bond_stack.empty:
    print(f'Bond stack: {len(hca_bond_stack)} bonds')
    print(f'Outstanding bonds: {(~hca_bond_stack["is_matured"].fillna(True)).sum()}')
    wac = compute_weighted_avg_coupon(hca_bond_stack)
    print(f'Weighted Average Coupon (WAC): {wac:.2f}%')
    display(hca_bond_stack[~hca_bond_stack['is_matured'].fillna(True)].sort_values('maturity_date').head(10))
else:
    print('No TRACE data available — XBRL-only mode')

In [ ]:
print('--- TRACE Data Coverage Analysis ---')
print('What fraction of the universe has TRACE enrichment?\n')

trace_coverage = []
for _, co in universe_df[universe_df['include_flag'] == 1].head(10).iterrows():
    cusips = build_cusip_list_from_edgar(str(co['cik']).zfill(10))
    trace_coverage.append({
        'ticker': co['ticker'],
        'n_cusips': len(cusips),
        'trace_available': len(cusips) > 0
    })

cov_df = pd.DataFrame(trace_coverage)
display(cov_df)
print(f'\nTRACE available: {cov_df["trace_available"].sum()}/{len(cov_df)} companies')

---
## Section 6: Signal Component Analysis

Three building blocks of the composite stress signal:
1. **MCS** — how concentrated is the maturity profile?
2. **RDM** — how expensive is today's refinancing relative to issuance?
3. **decay_weight** — how close is the peak maturity date?

In [ ]:
from src.features import (
    compute_maturity_concentration_score,
    compute_months_to_peak_maturity,
    compute_signal_decay_weight,
)
from src.data_loader import fetch_all_fred_series

macro_rates = fetch_all_fred_series()
print(f'Macro rates loaded: {len(macro_rates)} days × {len(macro_rates.columns)} series')
print(f'Date range: {macro_rates.index.min().date()} to {macro_rates.index.max().date()}')
display(macro_rates.tail(5))

In [ ]:
# MCS for HCA as of a single date
as_of = pd.Timestamp('2023-03-31')
mcs_snap = compute_maturity_concentration_score(hca_quarterly, as_of)
print(f'MCS as of {as_of.date()}:')
display(mcs_snap)

In [ ]:
from src.visualize import plot_signal_decay_function

fig = plot_signal_decay_function()
plt.show()
print('The decay function is the mathematical answer to the carry-bleed problem.'
      '\nAt 36 months: weight ≈ 0.25 (distant — low exposure)'
      '\nAt 18 months: weight ≈ 1.00 (medium — neutral)'
      '\nAt  6 months: weight ≈ 2.00 (near — doubled)'
      '\nAt  0 months: weight ≈ 4.00 (imminent — maximum)')

---
## Section 7: Macro Rate Overlay

The RDM requires current market rates. We use FRED free series:
- `DGS10` = 10-year Treasury (risk-free rate)
- `BAMLH0A0HYM2` = ICE BofA HY OAS (high-yield credit spread)
- `BAMLC0A0CM` = ICE BofA IG OAS (investment-grade spread)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

macro_plot = macro_rates['2014':]

# Treasury rates
axes[0].plot(macro_plot.index, macro_plot['T10Y'], color='#2C3E50', linewidth=1.5, label='10Y Treasury')
axes[0].plot(macro_plot.index, macro_plot['T2Y'], color='#3498DB', linewidth=1, linestyle='--', label='2Y Treasury')
axes[0].set_ylabel('Yield (%)', fontsize=11)
axes[0].set_title('FRED Rate Series: Treasury Yields and Credit Spreads', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].axhline(0, color='gray', linewidth=0.5)

# Credit spreads
axes[1].fill_between(macro_plot.index, macro_plot['HY_OAS'], alpha=0.3, color='#E74C3C')
axes[1].plot(macro_plot.index, macro_plot['HY_OAS'], color='#C0392B', linewidth=1.5, label='HY OAS (bps)')
axes[1].plot(macro_plot.index, macro_plot['IG_OAS'], color='#27AE60', linewidth=1.5, label='IG OAS (bps)')
axes[1].axhline(600, color='red', linestyle=':', linewidth=1, label='WIDE_SPREADS threshold (600bps)')
axes[1].axhline(350, color='green', linestyle=':', linewidth=1, label='TIGHT_SPREADS threshold (350bps)')
axes[1].set_ylabel('OAS (basis points)', fontsize=11)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print('\nMacro regime periods in backtest history (2014–2023):')
print('  2014–2018: Low-rate era — RDM low, carry bleed more severe')
print('  2020 Q1:   COVID spike — HY_OAS > 900bps; WIDE_SPREADS regime')
print('  2022–2023: Rapid rate rise — RDM spikes; RISING_RATES regime')

---
## Section 8: Signal Validation — Historical Stress Events

The 15 documented stress events in `historical_stress_events.csv` are the strategy's ground-truth validation dataset.  
For each event: did the signal fire ahead of time? How many months of advance warning?

In [ ]:
stress_events = pd.read_csv(config.HISTORICAL_STRESS_EVENTS_CSV)
print(f'Historical stress events: {len(stress_events)}')
display(stress_events[['ticker', 'event_type', 'stress_start_date', 
                        'maturity_wall_months_ahead_at_signal', 
                        'signal_would_have_triggered', 'notes']].to_string(index=False))

In [ ]:
print('Signal trigger rate (from reference data):')
trigger_rate = stress_events['signal_would_have_triggered'].mean()
print(f'  {trigger_rate:.0%} of documented stress events would have triggered CRITICAL/ELEVATED')
print(f'  Median months of advance warning: {stress_events["maturity_wall_months_ahead_at_signal"].median():.0f} months')
print(f'  Min lead time: {stress_events["maturity_wall_months_ahead_at_signal"].min():.0f} months')
print(f'  Max lead time: {stress_events["maturity_wall_months_ahead_at_signal"].max():.0f} months')

print('\nEvent type distribution:')
display(stress_events['event_type'].value_counts())

In [ ]:
# If rolling signals are available, run formal lead-time analysis
if config.ROLLING_SIGNALS_PARQUET.exists():
    from src.signal import compute_signal_lead_time_analysis
    from src.visualize import plot_lead_time_distribution
    
    rolling_signals = pd.read_parquet(config.ROLLING_SIGNALS_PARQUET)
    lead_time_df = compute_signal_lead_time_analysis(rolling_signals, stress_events)
    
    print('--- Lead Time Analysis Results ---')
    display(lead_time_df)
    
    fig = plot_lead_time_distribution(lead_time_df)
    plt.show()
else:
    # Show reference-data lead times as proxy
    from src.visualize import plot_lead_time_distribution
    lead_time_proxy = stress_events[stress_events['signal_would_have_triggered'] == 1].copy()
    lead_time_proxy = lead_time_proxy.rename(columns={
        'maturity_wall_months_ahead_at_signal': 'lead_time_months',
        'signal_would_have_triggered': 'signal_triggered'
    })
    fig = plot_lead_time_distribution(lead_time_proxy)
    plt.show()
    print('\n(Using reference-data lead times — run full pipeline for model-derived lead times)')

---
## Section 9: Statistical Tests

In [ ]:
if config.ROLLING_SIGNALS_PARQUET.exists() and config.EQUITY_PRICES_PARQUET.exists():
    from src.signal import run_ttest_validation, compute_information_coefficient, compute_sector_breakdown
    
    rolling_signals = pd.read_parquet(config.ROLLING_SIGNALS_PARQUET)
    equity_prices   = pd.read_parquet(config.EQUITY_PRICES_PARQUET)
    
    # t-test
    ttest_results = run_ttest_validation(rolling_signals, equity_prices)
    
    # IC analysis
    ic_results = compute_information_coefficient(rolling_signals, equity_prices)
    
    # Sector breakdown (requires sector column from universe merge)
    signals_with_sector = rolling_signals.merge(
        universe_df[['ticker', 'sector']], on='ticker', how='left'
    )
    sector_ic = compute_sector_breakdown(signals_with_sector, equity_prices)
    print('\n--- IC by Sector ---')
    display(sector_ic)
    
    from src.visualize import plot_ic_by_horizon_and_regime
    fig = plot_ic_by_horizon_and_regime(ic_results)
    plt.show()
else:
    print('Full pipeline not yet run. Run build_rolling_signal_history() first.')
    print('\nExpected results (from reference data cross-check):')
    print('  IC at 30d:  ~0.05–0.10 (weak — equity does not move immediately)')
    print('  IC at 180d: ~0.15–0.25 (stronger — market reprices over 6 months)')
    print('  IC at 252d: ~0.20–0.30 (strongest — the annual-horizon thesis holds)')

---
## Section 10: Backtest Results

In [ ]:
if config.ROLLING_SIGNALS_PARQUET.exists() and config.EQUITY_PRICES_PARQUET.exists():
    from src.backtest import (
        run_short_only_backtest, run_long_short_backtest,
        run_decay_weight_comparison, run_stress_event_overlay,
        compute_performance_metrics
    )
    from src.visualize import plot_cumulative_pnl_with_carry_bleed
    
    rolling_signals = pd.read_parquet(config.ROLLING_SIGNALS_PARQUET)
    equity_prices   = pd.read_parquet(config.EQUITY_PRICES_PARQUET)
    
    # Short-only
    bt_short = run_short_only_backtest(rolling_signals, equity_prices)
    metrics_short = compute_performance_metrics(bt_short, label='Short-Only')
    
    # Long-short
    bt_ls = run_long_short_backtest(rolling_signals, equity_prices)
    metrics_ls = compute_performance_metrics(bt_ls, label='Long-Short')
    
    # Decay comparison
    comparison = run_decay_weight_comparison(rolling_signals, equity_prices)
    
    # Event overlay
    event_overlay = run_stress_event_overlay(bt_ls, stress_events)
    
    # PnL chart
    fig = plot_cumulative_pnl_with_carry_bleed(bt_ls, event_overlay)
    plt.show()
    
    # Carry-bleed quantification
    print('\n--- Carry Bleed Analysis by Event ---')
    display(event_overlay[['ticker', 'event_type', 'carry_bleed_return', 'payoff_return', 'net_event_return']])
    
    # Break-even analysis
    borrow_base  = 0.0050  # 50bps base case
    borrow_high  = 0.0150  # 150bps stress case
    borrow_severe = 0.0300  # 300bps severe case
    print('\n--- Short Borrow Cost Break-Even Analysis ---')
    for borrow_annual, label in [(borrow_base, '50bps (base)'), (borrow_high, '150bps (stress)'), (borrow_severe, '300bps (severe)')]:
        borrow_q = borrow_annual / 4
        required_gross = borrow_q + 0.0015  # + tc
        print(f'  Borrow @ {label}: need >{required_gross:.2%} quarterly gross short return to break even')
else:
    print('Run the full pipeline first: build_rolling_signal_history()')

---
## Section 11: Limitations & Extensions

### Permanent Limitations

**1. XBRL Tag Inconsistency**  
XBRL tag inconsistency is not a solvable problem — it is a permanent feature of SEC disclosure. Companies change auditors and tagging conventions across years. Manual taxonomy maintenance is required.  

*Minimum quarterly maintenance burden*: 2–4 hours to review new extension tags from companies that filed 10-Ks in the quarter. Each new tag must be mapped to a standard bucket and added to `xbrl_tag_taxonomy.csv`.

**2. The Signal-Is-Right-But-Early Problem**  
A company can refinance, extend, or get acquired in month 17 of an 18-month stress window. The strategy has no mechanism to detect these resolutions in real time.  

*Suggested extension*: Add a **Resolution Monitor** that watches for new bond issuance (8-K filings with bond prospectus — specifically, filings that include `424B2` or `424B5` form types tagged with the company's CIK) as a signal to close the short. A successful refinancing typically appears in EDGAR within 5 business days of closing.

**3. Short Borrow Availability**  
The most stressed names are hardest to borrow. The 50bps borrow cost assumption may be optimistic for CCC-rated issuers with known distress — actual borrow costs can be 200–500bps for hard-to-borrow names. Stress-tested at 150bps and 300bps above.

**4. RDM Coupon Proxy Imprecision**  
The RDM uses coupon rate as a proxy for yield-at-issuance. This is incorrect for:  
- Discount bonds (issued below par): YTM > coupon → RDM understates difficulty  
- Premium bonds: YTM < coupon → RDM overstates difficulty  
The true YTM at issuance requires the original issue price from the bond prospectus.

**5. Macro-Regime Concentration**  
If the strategy's Sharpe ratio is dramatically higher in rising-rate periods, it may be a disguised rate-sensitivity bet. Test: regress strategy returns on T10Y changes. If R² > 0.5, the signal is more rate-bet than maturity-wall bet.

**6. Survivorship Bias**  
The universe is constructed from companies known to the researcher ex-post. A live implementation would use a contemporaneous high-yield index membership (e.g. ICE BofA HY Index constituents at each quarter-end) as the entry criterion.

### Natural Extensions

**Leveraged Loans**  
Apply the same XBRL maturity wall analysis to leveraged loan borrowers. LSTA loan reference data provides outstanding amounts and spread levels. CLO manager reinvestment constraints create additional mechanical selling pressure near loan maturity — a second source of alpha beyond the fundamental refinancing difficulty.

**Municipal Bonds**  
MSRB Electronic Municipal Market Access (EMMA) provides bond reference data with maturity schedules for municipal issuers. Same framework applies to hospital systems and utility districts with concentrated near-term maturities.

**Real-Time Resolution Monitor**  
Build an EDGAR webhook listener that watches for 8-K filings from portfolio shorts. Any 8-K containing 'completed', 'issued', 'closed' AND 'notes' or 'bonds' in the body should trigger a signal-review alert. This converts the strategy from a quarterly-rebalanced backtest into a continuous monitoring system.

In [ ]:
print('=== Notebook Complete ===')
print('\nAll charts saved to:', config.OUTPUTS_DIR)
print('All processed data cached to:', config.PROCESSED_DIR)
print('\nNext steps:')
print('  1. Run: python -m src.xbrl_parser --cik 0000860731  (smoke test)')
print('  2. Run: build_universe_maturity_schedules() for full coverage')
print('  3. Run: build_rolling_signal_history() to generate the backtest panel')
print('  4. Run all Section 9-10 cells for statistical results')